# Module End Assignment 4 – Machine Learning: Car Price Prediction

**Objective:** Predict car prices and identify the variables associated with price using the dataset specified in the assignment.

This notebook covers:
1. Dataset loading and preprocessing
2. Linear Regression, Decision Tree, Random Forest, Gradient Boosting, and Support Vector Regression
3. Evaluation using R², MSE, and MAE
4. Feature importance
5. Hyperparameter tuning and before/after comparison

**Dataset source provided in the assignment:** [Google Drive dataset](https://drive.google.com/file/d/1FHmYNLs9v0Enc-UExEMpitOFGsWvB2dP/view?usp=drive_link)

Run the cells from top to bottom. If automatic download does not work, upload the CSV file when prompted. Results are generated from the actual dataset when you run the notebook; they are not hard-coded.

In [ ]:
# If needed, install the libraries used in this notebook
%pip -q install pandas numpy scikit-learn matplotlib gdown

In [ ]:
import os
import glob
import subprocess
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
print("Libraries imported successfully.")

In [ ]:
# Load the assignment dataset.
# The notebook first checks the current folder, then tries downloading from the provided Google Drive link.
csv_candidates = glob.glob("*.csv") + glob.glob("/content/*.csv")
csv_candidates = list(dict.fromkeys(csv_candidates))
df = None

if csv_candidates:
    # Prefer the common dataset name if present; otherwise use the first CSV found.
    preferred = [p for p in csv_candidates if "carprice" in os.path.basename(p).lower() or "car_price" in os.path.basename(p).lower()]
    dataset_path = preferred[0] if preferred else csv_candidates[0]
    df = pd.read_csv(dataset_path)
    print(f"Loaded dataset: {dataset_path}")
else:
    try:
        import gdown
        file_id = "1FHmYNLs9v0Enc-UExEMpitOFGsWvB2dP"
        output_path = "CarPrice_Assignment.csv"
        downloaded = gdown.download(id=file_id, output=output_path, quiet=False, fuzzy=True)
        if downloaded and os.path.exists(output_path) and os.path.getsize(output_path) > 0:
            df = pd.read_csv(output_path)
            print(f"Downloaded and loaded: {output_path}")
    except Exception as e:
        print("Automatic download was not successful:", e)

if df is None:
    print("Please upload the assignment CSV file below.")
    try:
        from google.colab import files
        uploaded = files.upload()
        uploaded_csv = next((name for name in uploaded if name.lower().endswith(".csv")), None)
        if uploaded_csv:
            df = pd.read_csv(uploaded_csv)
    except Exception:
        pass

if df is None:
    raise FileNotFoundError(
        "Dataset not found. Download the CSV from the Google Drive link in the assignment, "
        "place it in the same folder as this notebook, and rerun this cell."
    )

print("Dataset shape:", df.shape)
display(df.head())
display(df.info())

## 1. Loading and preprocessing

The target variable is `price`. The code removes common identifier columns when present, separates features from the target, imputes missing values, one-hot encodes categorical features, and scales numeric features for the Support Vector Regressor. Rows with missing target prices are removed.

In [ ]:
# Normalize column labels only by removing surrounding whitespace.
df.columns = [str(c).strip() for c in df.columns]

if "price" not in df.columns:
    raise ValueError(
        "A target column named 'price' was not found. Please check that you uploaded the correct car-price dataset. "
        f"Available columns: {list(df.columns)}"
    )

df = df.dropna(subset=["price"]).copy()
# Remove identifier-like columns that do not represent meaningful car characteristics.
id_cols = [c for c in df.columns if c.lower() in {"car_id", "carid", "id", "unnamed: 0"}]
df = df.drop(columns=id_cols, errors="ignore")

y = pd.to_numeric(df["price"], errors="coerce")
valid_target = y.notna()
df = df.loc[valid_target].copy()
y = y.loc[valid_target]
X = df.drop(columns=["price"]).copy()

# Ensure categorical values are treated as strings and numeric columns remain numeric.
for col in X.columns:
    if X[col].dtype == "object":
        X[col] = X[col].astype("string").fillna(np.nan)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_preprocessor, numeric_features),
    ("cat", categorical_preprocessor, categorical_features)
])

print("Rows after cleaning:", len(X))
print("Number of input features before encoding:", X.shape[1])
print("Numeric columns:", len(numeric_features))
print("Categorical columns:", len(categorical_features))
print("Missing values in target after cleaning:", int(y.isna().sum()))
display(pd.DataFrame({
    "Column": X.columns,
    "Data type": [str(X[c].dtype) for c in X.columns],
    "Missing values": [int(X[c].isna().sum()) for c in X.columns]
}))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print(f"Training rows: {X_train.shape[0]}")
print(f"Testing rows: {X_test.shape[0]}")

## 2. Implement the five regression models

All models use the same training/test split and preprocessing pipeline so their scores can be compared fairly.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "Support Vector Regressor": SVR(kernel="rbf", C=100, epsilon=0.1)
}

results = []
fitted_models = {}

for name, estimator in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    predictions = pipe.predict(X_test)
    results.append({
        "Model": name,
        "R-squared (R²)": r2_score(y_test, predictions),
        "MSE": mean_squared_error(y_test, predictions),
        "MAE": mean_absolute_error(y_test, predictions)
    })
    fitted_models[name] = pipe

results_df = pd.DataFrame(results).sort_values(
    by="R-squared (R²)", ascending=False
).reset_index(drop=True)
display(results_df.style.format({
    "R-squared (R²)": "{:.4f}", "MSE": "{:,.2f}", "MAE": "{:,.2f}"
}))
best_baseline_name = results_df.iloc[0]["Model"]
print("Best baseline model by R²:", best_baseline_name)
print("A good model has higher R² and lower MSE and MAE. R² measures explained variation; MSE penalizes large errors more strongly; MAE is the average absolute prediction error.")

In [ ]:
# Visual comparison of model scores
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
metrics = [("R-squared (R²)", "R² (higher is better)"),
           ("MSE", "MSE (lower is better)"),
           ("MAE", "MAE (lower is better)")]
for ax, (column, title) in zip(axes, metrics):
    ordered = results_df.sort_values(column, ascending=(column != "R-squared (R²)"))
    ax.barh(ordered["Model"], ordered[column])
    ax.set_title(title)
    ax.set_xlabel(column)
    ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.show()

## 3. Identify the best-performing model

The table above is calculated from the test data. The notebook selects the highest test-set R² as the primary ranking criterion, while also considering MSE and MAE. If a model has the highest R² but much worse error values, interpret the metrics together rather than relying on one score alone.

In [ ]:
best_row = results_df.iloc[0]
print(f"Best baseline model by test R²: {best_row['Model']}")
print(f"Test R²: {best_row['R-squared (R²)']:.4f}")
print(f"Test MSE: {best_row['MSE']:.2f}")
print(f"Test MAE: {best_row['MAE']:.2f}")
print("\nConclusion: this model explains the largest share of test-set price variation among the five baseline models. Use the error metrics above to describe the typical size of prediction errors.")

## 4. Feature importance / significant variables

Tree-based models provide feature-importance scores. The code extracts the transformed feature names (including one-hot encoded categories) and ranks the most influential features. Importance indicates contribution to the model's predictions; it does not by itself prove causation or statistical significance.

In [ ]:
# Use the best-performing tree-based baseline model for feature importance.
tree_names = ["Decision Tree Regressor", "Random Forest Regressor", "Gradient Boosting Regressor"]
tree_results = results_df[results_df["Model"].isin(tree_names)]
if len(tree_results) > 0:
    chosen_tree_name = tree_results.iloc[0]["Model"]
    chosen_tree_pipe = fitted_models[chosen_tree_name]
    fitted_preprocessor = chosen_tree_pipe.named_steps["preprocessor"]
    fitted_tree = chosen_tree_pipe.named_steps["model"]
    feature_names = fitted_preprocessor.get_feature_names_out()
    importances = getattr(fitted_tree, "feature_importances_", None)
    if importances is not None and len(importances) == len(feature_names):
        importance_df = pd.DataFrame({
            "Feature": feature_names,
            "Importance": importances
        }).sort_values("Importance", ascending=False).reset_index(drop=True)
        print("Feature importance from:", chosen_tree_name)
        display(importance_df.head(20).style.format({"Importance": "{:.5f}"}))
        top_features = importance_df.head(15).sort_values("Importance")
        plt.figure(figsize=(10, 6))
        plt.barh(top_features["Feature"], top_features["Importance"])
        plt.xlabel("Feature importance")
        plt.title(f"Top features – {chosen_tree_name}")
        plt.tight_layout()
        plt.show()
        print("The top rows in the table are the variables most influential for this fitted tree-based model.")
    else:
        print("Feature importance could not be extracted for the selected model.")
else:
    print("No tree-based model was available for feature-importance analysis.")

## 5. Hyperparameter tuning

Grid search with cross-validation tunes a Random Forest Regressor using training data only. The tuned model is then evaluated on the same held-out test set and compared with the baseline Random Forest. The test set is not used to choose the hyperparameters.

In [ ]:
baseline_rf = fitted_models["Random Forest Regressor"]
baseline_rf_pred = baseline_rf.predict(X_test)
baseline_rf_metrics = {
    "Model": "Random Forest (baseline)",
    "R²": r2_score(y_test, baseline_rf_pred),
    "MSE": mean_squared_error(y_test, baseline_rf_pred),
    "MAE": mean_absolute_error(y_test, baseline_rf_pred)
}

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
])
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_leaf": [1, 2]
}
grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)
tuned_rf = grid_search.best_estimator_
tuned_pred = tuned_rf.predict(X_test)
tuned_rf_metrics = {
    "Model": "Random Forest (tuned)",
    "R²": r2_score(y_test, tuned_pred),
    "MSE": mean_squared_error(y_test, tuned_pred),
    "MAE": mean_absolute_error(y_test, tuned_pred)
}

tuning_comparison = pd.DataFrame([baseline_rf_metrics, tuned_rf_metrics])
print("Best parameters:", grid_search.best_params_)
print(f"Best cross-validation R²: {grid_search.best_score_:.4f}")
display(tuning_comparison.style.format({"R²": "{:.4f}", "MSE": "{:,.2f}", "MAE": "{:,.2f}"}))

r2_change = tuned_rf_metrics["R²"] - baseline_rf_metrics["R²"]
mse_change = tuned_rf_metrics["MSE"] - baseline_rf_metrics["MSE"]
mae_change = tuned_rf_metrics["MAE"] - baseline_rf_metrics["MAE"]
if r2_change > 0 and mse_change < 0 and mae_change < 0:
    print("Conclusion: tuning improved all three test metrics.")
elif r2_change > 0 or mse_change < 0 or mae_change < 0:
    print("Conclusion: tuning improved some metrics, but not all. Review the comparison table.")
else:
    print("Conclusion: tuning did not improve these test metrics for this split. The tuned parameters were selected by cross-validation, but improvement is not guaranteed.")

## 6. Final answers to the business questions

- **Which variables are significant?** Review the feature-importance table above. The top-ranked features are the most influential predictors according to the selected tree-based model.
- **How well do the variables describe price?** Use the test-set R² for the selected baseline model and the MSE/MAE values to quantify prediction error.
- **Which model is best?** The first row of the sorted baseline evaluation table is the highest-R² model on the held-out test set. Check its MSE and MAE before making the final judgement.
- **Did tuning improve performance?** Compare the baseline and tuned Random Forest results in the tuning table. The printed conclusion reports whether the metrics improved.

**Important:** Actual scores and top variables depend on the provided CSV and are displayed only after running the notebook.